# Attack Success Rate Evaluation Using BERTAttack — Multi-Model

Evaluates 3 models sequentially. Results are saved under:
```
RESULTS/<model_name>/adv_success_test.jsonl
RESULTS/<model_name>/progress_test.json
RESULTS/<model_name>/summary.json
```

## Configuration
- **Attack**: BERTAttack (Li et al., 2020)
- **USE Threshold**: 0.8
- **Max Perturbation**: 10% of words
- **Query Budget**: 80 per example

In [1]:
# Imports
import os, gc, json, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
import textattack
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import BERTAttackLi2020
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.datasets import Dataset as TADataset
from textattack import Attacker
from textattack.attack_args import AttackArgs

SEED = 2026
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 0. Global Configuration
Edit `MODEL_DIRS` to point to your 3 model directories.

In [9]:
# ── Edit these ──────────────────────────────────────────────────────────────
MODEL_DIRS = [
    './baseline_42',
    'adversarial_trained_model_42.25/final_model',
    'adversarial_trained_model_42.50/final_model2',
      './baseline_123',
    'adversarial_trained_model_123.25/final_model3',
    'adversarial_trained_model_123.50/final_model4',
      './baseline_2026',
    'adversarial_trained_model_2026.25/final_model5',
    'adversarial_trained_model_2026.50/final_model6',
]
# ────────────────────────────────────────────────────────────────────────────

RESULTS_ROOT = 'RESULTS'          # top-level output folder
SPLIT        = 'test'
N_EXAMPLES   = 1283
MAX_LENGTH   = 128
USE_THRESHOLD  = 0.8
MAX_PCT_WORDS  = 0.10
MAX_CANDIDATES = 5
QUERY_BUDGET   = 80
BATCH_SIZE     = 1282

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Models : {MODEL_DIRS}')
print(f'Output : {RESULTS_ROOT}/<model_name>/')

Device : cuda
Models : ['./baseline_42', 'adversarial_trained_model_42.25/final_model', 'adversarial_trained_model_42.50/final_model2', './baseline_123', 'adversarial_trained_model_123.25/final_model3', 'adversarial_trained_model_123.50/final_model4', './baseline_2026', 'adversarial_trained_model_2026.25/final_model5', 'adversarial_trained_model_2026.50/final_model6']
Output : RESULTS/<model_name>/


## 1. Load & Prepare Dataset (once)

In [3]:
ds = load_dataset('ucsbnlp/liar', trust_remote_code=True)
print({k: len(ds[k]) for k in ds.keys()})

{'train': 10269, 'test': 1283, 'validation': 1284}


In [4]:
LABELS   = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def safe_str(x):
    if x is None: return ''
    if isinstance(x, (list, tuple)):
        x = ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject   = safe_str(example.get('subjects', example.get('subject', '')))
    context   = safe_str(example.get('context', ''))
    parts     = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    example['label_id'] = label2id[y.strip().lower()] if isinstance(y, str) else int(y)
    return example

ds = ds.map(build_text)

# Build the shared list of (text, label_id) pairs once
split_ds = ds[SPLIT].select(range(min(N_EXAMPLES, len(ds[SPLIT]))))
ta_pairs = [(ex['text'], ex['label_id']) for ex in split_ds]
ta_pairs.pop(990)
print(f'Test pairs ready: {len(ta_pairs)}')
print(f"Example: {ta_pairs[0][0][:120]}...")

Test pairs ready: 1282
Example: Building a wall on the U.S.-Mexico border will take literally years. [SEP] SUBJECT: immigration [SEP] CONTEXT: Radio int...


## 2. Helper Functions

In [5]:
def get_model_out_dir(model_dir):
    """Return RESULTS/<model_name>/ path and create it."""
    model_name = os.path.basename(model_dir.rstrip('/\\'))
    out_dir    = os.path.join(RESULTS_ROOT, model_name)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir, model_name

def save_state(state_path, next_index, success_count):
    tmp = state_path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump({'next_index': next_index, 'success_count': success_count}, f)
    os.replace(tmp, state_path)

def append_success(jsonl_path, rows):
    with open(jsonl_path, 'a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def build_attack(model, tokenizer):
    """Build and configure BERTAttack for a given model wrapper."""
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    attack = BERTAttackLi2020.build(model_wrapper)

    for c in attack.constraints:
        if c.__class__.__name__ == 'UniversalSentenceEncoder':
            c.threshold = USE_THRESHOLD

    attack.transformation.max_length = MAX_LENGTH
    if hasattr(attack.transformation, 'max_candidates'):
        attack.transformation.max_candidates = MAX_CANDIDATES

    attack.constraints = [
        c for c in attack.constraints
        if c.__class__.__name__ not in ('MaxPercentWordsPerturbed', 'MaxWordsPerturbed')
    ]
    attack.constraints.append(MaxWordsPerturbed(max_percent=MAX_PCT_WORDS))
    return attack

print('Helpers defined.')

Helpers defined.


## 3. Evaluate Each Model

In [6]:
all_summaries = []

for model_dir in MODEL_DIRS:
    out_dir, model_name = get_model_out_dir(model_dir)
    state_path = os.path.join(out_dir, f'progress_{SPLIT}.json')
    jsonl_path = os.path.join(out_dir, f'adv_success_{SPLIT}.jsonl')

    print('=' * 60)
    print(f'MODEL : {model_name}')
    print(f'OUTPUT: {out_dir}')
    print('=' * 60)

    # ── Load model ──────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, model_max_length=MAX_LENGTH, use_fast=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.config.id2label = id2label
    model.config.label2id = label2id
    model.to(device)
    model.eval()
    torch.set_grad_enabled(False)
    print(f'Loaded model | num_labels: {model.config.num_labels}')

    # ── Build attack ────────────────────────────────────────────
    attack = build_attack(model, tokenizer)

    # ── Resume or start fresh ───────────────────────────────────
    start_idx, success_count = 0, 0
    if os.path.exists(state_path):
        with open(state_path, 'r', encoding='utf-8') as f:
            st = json.load(f)
        start_idx     = int(st.get('next_index', 0))
        success_count = int(st.get('success_count', 0))
        print(f'Resuming from index {start_idx} (success so far: {success_count})')
    else:
        print('Starting fresh.')

    # ── Run batched attack ───────────────────────────────────────
    total = len(ta_pairs)
    for i in range(start_idx, total, BATCH_SIZE):
        j          = min(i + BATCH_SIZE, total)
        batch_pairs = ta_pairs[i:j]
        batch_data  = TADataset(batch_pairs)

        attack_args = AttackArgs(
            num_examples=len(batch_data),
            query_budget=QUERY_BUDGET,
            random_seed=SEED,
            disable_stdout=True,
            log_to_csv=None
        )

        print(f'  Batch [{i}:{j}] ...')
        attacker      = Attacker(attack, batch_data, attack_args)
        batch_results = attacker.attack_dataset()

        rows = []
        for k, r in enumerate(batch_results):
            if r.__class__.__name__ == 'SuccessfulAttackResult':
                rows.append({
                    'text'       : r.perturbed_result.attacked_text.text,
                    'label'      : int(r.original_result.ground_truth_output),
                    'orig_index' : i + k
                })

        success_count += len(rows)
        if rows:
            append_success(jsonl_path, rows)
        save_state(state_path, next_index=j, success_count=success_count)

        batch_asr = len(rows) / len(batch_data) * 100 if batch_data else 0
        print(f'  Batch: {len(rows)}/{len(batch_data)} ({batch_asr:.1f}%)')
        print(f'  Running total: {success_count}/{j} ({success_count/j*100:.1f}%)')

        del attacker, batch_results, batch_data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Save summary ─────────────────────────────────────────────
    asr = success_count / total * 100
    summary = {
        'model'              : model_name,
        'model_dir'          : model_dir,
        'total_evaluated'    : total,
        'successful_attacks' : success_count,
        'attack_success_rate': round(asr, 4),
        'model_robustness'   : round(100 - asr, 4),
        'jsonl_path'         : jsonl_path,
    }
    summary_path = os.path.join(out_dir, 'summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    all_summaries.append(summary)
    print(f'\n  ✓ {model_name}: ASR = {asr:.2f}%  |  Robustness = {100-asr:.2f}%')
    print(f'  Summary saved → {summary_path}\n')

    # Free GPU memory before next model
    del model, tokenizer, attack
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n' + '=' * 60)
print('ALL MODELS COMPLETE')
print('=' * 60)

MODEL : baseline_2026
OUTPUT: RESULTS\baseline_2026
Loaded model | num_labels: 6


C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
text

Starting fresh.
  Batch [0:1282] ...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapMaskedLM(
    (method):  bert-attack
    (masked_lm_name):  BertForMaskedLM
    (max_length):  128
    (max_candidates):  5
    (min_confidence):  0.0005
  )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  cosine
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): MaxWordsPerturbed(
        (max_percent):  0.1
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 0 / 0 / 2 / 2:   0%|                           | 2/1282 [00:00<02:06, 10.13it/s]


[Succeeded / Failed / Skipped / Total] 348 / 46 / 888 / 1282: 100%|████████████████| 1282/1282 [03:12<00:00,  6.65it/s]

...

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
textattack: Unknown if model of class <class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2ForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Starting fresh.
  Batch [0:1282] ...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapMaskedLM(
    (method):  bert-attack
    (masked_lm_name):  BertForMaskedLM
    (max_length):  128
    (max_candidates):  5
    (min_confidence):  0.0005
  )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  cosine
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): MaxWordsPerturbed(
        (max_percent):  0.1
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 352 / 43 / 887 / 1282: 100%|████████████████| 1282/1282 [02:57<00:00,  7.21it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 352    |
| Number of failed attacks:     | 43     |
| Number of skipped attacks:    | 887    |
| Original accuracy:            | 30.81% |
| Accuracy under attack:        | 3.35%  |
| Attack success rate:          | 89.11% |
| Average perturbed word %:     | 5.13%  |
| Average num. words per input: | 28.54  |
| Avg num queries:              | 29.03  |
+-------------------------------+--------+


  Batch: 352/1282 (27.5%)
  Running total: 352/1282 (27.5%)

  ✓ final_model6: ASR = 27.46%  |  Robustness = 72.54%
  Summary saved → RESULTS\final_model6\summary.json


ALL MODELS COMPLETE


## 4. Comparison Table

In [10]:
# Load summaries (works even if run in a fresh kernel after evaluation)
loaded_summaries = []
for model_dir in MODEL_DIRS:
    model_name   = os.path.basename(model_dir.rstrip('/\\'))
    summary_path = os.path.join(RESULTS_ROOT, model_name, 'summary.json')
    if os.path.exists(summary_path):
        with open(summary_path, 'r', encoding='utf-8') as f:
            loaded_summaries.append(json.load(f))
    else:
        print(f'WARNING: summary not found for {model_name}')

cmp_df = pd.DataFrame(loaded_summaries)[[
    'model', 'total_evaluated', 'successful_attacks',
    'attack_success_rate', 'model_robustness'
]]
cmp_df.columns = ['Model', 'Evaluated', 'Successful Attacks', 'ASR (%)', 'Robustness (%)']
display(cmp_df)

# Save comparison CSV
os.makedirs(RESULTS_ROOT, exist_ok=True)
csv_path = os.path.join(RESULTS_ROOT, 'comparison_all.csv')
cmp_df.to_csv(csv_path, index=False)
print(f'\nComparison saved → {csv_path}')

,Model,Evaluated,Successful Attacks,ASR (%),Robustness (%)
0,baseline_42,1282,368,28.7051,71.2949
1,final_model,1282,327,25.5070,74.4930
2,final_model2,1282,308,24.0250,75.9750
3,baseline_123,1282,370,28.8612,71.1388
4,final_model3,1282,366,28.5491,71.4509
5,final_model4,1282,373,29.0952,70.9048
6,baseline_2026,1282,348,27.1451,72.8549
7,final_model5,1282,346,26.9891,73.0109
8,final_model6,1282,352,27.4571,72.5429



Comparison saved → RESULTS\comparison_all.csv


## 5. Per-Model Detailed Analysis

In [8]:
for model_dir in MODEL_DIRS:
    model_name = os.path.basename(model_dir.rstrip('/\\'))
    jsonl_path = os.path.join(RESULTS_ROOT, model_name, f'adv_success_{SPLIT}.jsonl')

    rows = []
    if os.path.exists(jsonl_path):
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))

    if not rows:
        print(f'{model_name}: no successful attacks recorded.\n')
        continue

    df = pd.DataFrame(rows)
    print(f'── {model_name} ──────────────────────────────')
    print(f'Total successful attacks: {len(df)}')
    print('Label distribution:')
    for label_id, count in df['label'].value_counts().sort_index().items():
        pct = count / len(df) * 100
        print(f'  {id2label[label_id]:15s}: {count:4d}  ({pct:.1f}%)')
    print('Sample adversarial examples:')
    display(df[['orig_index', 'label', 'text']].head(3))
    print()

── baseline_2026 ──────────────────────────────
Total successful attacks: 348
Label distribution:
  false          :   48  (13.8%)
  half-true      :   65  (18.7%)
  mostly-true    :   48  (13.8%)
  true           :   99  (28.4%)
  barely-true    :   63  (18.1%)
  pants-fire     :   25  (7.2%)
Sample adversarial examples:


,orig_index,label,text
0,2,0,Says John McCain has done no to help the vets....
1,5,3,Over the past five years the federal governmen...
2,6,3,the that nashville law requires that school re...



── final_model5 ──────────────────────────────
Total successful attacks: 346
Label distribution:
  false          :   48  (13.9%)
  half-true      :   70  (20.2%)
  mostly-true    :   91  (26.3%)
  true           :   31  (9.0%)
  barely-true    :   79  (22.8%)
  pants-fire     :   27  (7.8%)
Sample adversarial examples:


,orig_index,label,text
0,2,0,Says John McCain has done everything to help t...
1,4,5,When asked by a reporter how heс at the root o...
2,6,3,Says that Tennessee law requires that schools ...



── final_model6 ──────────────────────────────
Total successful attacks: 352
Label distribution:
  false          :   60  (17.0%)
  half-true      :   77  (21.9%)
  mostly-true    :  101  (28.7%)
  true           :   41  (11.6%)
  barely-true    :   57  (16.2%)
  pants-fire     :   16  (4.5%)
Sample adversarial examples:


,orig_index,label,text
0,2,0,Says John McCain has done everything to help t...
1,5,3,Over the past five years the federal governmen...
2,9,4,We know that more than half of Hillary Clinton...


## Notes

- Each model's outputs live in `RESULTS/<model_name>/`.
- `adv_success_test.jsonl` — adversarial examples (successful attacks only).
- `progress_test.json`    — checkpoint; delete to restart a model from scratch.
- `summary.json`          — ASR and robustness metrics.
- `comparison.csv`        — side-by-side comparison of all three models.
- All examples preserve semantic similarity (USE ≥ 0.8) and ≤ 10% word perturbation.